# BARseq data extraction 01

Extract flat files from the MATLAB package. No CCC models are created.

- Input: `data/BARseq/scripts/`.
- Output: `data/BARseq/extracted/`.
- Cohort: all neurons passing `nm1_qc_pass`; retain zero-target rows and all transcriptomic-quality flags.
- Run cells in order. Existing output files with the same names are replaced.
- Requires the project environment and `mat-io==0.4.2`. Install the reader in the notebook kernel with `%pip install mat-io==0.4.2` if needed.
- CSV files use UTF-8, no dataframe index, and `uid` for neuron joins. Read label columns with `keep_default_na=False` to preserve the literal `NA` label.

## Setup and source loading

In [1]:
import hashlib
import json
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from matio import load_from_mat

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
SOURCE_ROOT = REPO_ROOT / "data/BARseq/scripts"
OUTPUT_ROOT = REPO_ROOT / "data/BARseq/extracted"
MAT_PATH = SOURCE_ROOT / "data/proj_neurons_cohort.mat"
XCK12_PATH = SOURCE_ROOT / "data/xck_abc_mapping/xck12_harmony_predictions.csv.gz"
XCK17_PATH = SOURCE_ROOT / "data/xck_abc_mapping/xck17_harmony_predictions.csv.gz"
for path in (MAT_PATH, XCK12_PATH, XCK17_PATH):
    assert path.is_file(), f"Missing source file: {path}"

source = load_from_mat(str(MAT_PATH))
pn = source["pn"][0, 0]
source_meta = source["meta"][0, 0]
uid_all = pn["uid"].ravel()
qc = pn["nm1_qc_pass"].ravel()
assert qc.dtype == np.bool_ and qc.shape == uid_all.shape
assert pd.Index(uid_all).is_unique
assert pn["CCF"].shape == (len(uid_all), 3)
assert pn["proj_mat"].shape == (len(uid_all), 429)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
output_tables = {}


def write_table(frame: pd.DataFrame, filename: str) -> None:
    assert frame.columns.is_unique, f"Duplicate columns in {filename}"
    frame.to_csv(OUTPUT_ROOT / filename, index=False, encoding="utf-8")
    output_tables[filename] = {"rows": len(frame), "columns": frame.columns.tolist()}


print(f"QC cohort: {qc.sum():,}/{len(qc):,} neurons")

QC cohort: 1,448/1,512 neurons


## 1. `neurons.csv`

Source: `pn.uid`, `pn.brain`, QC flags, and `pn.cell_row`.

`full_row` indexes the unfiltered cohort; `cell_row` indexes the all-cells cache. Both retain MATLAB's one-based indexing.

In [2]:
neurons = pd.DataFrame({
    "uid": uid_all[qc],
    "brain": pn["brain"].ravel()[qc],
    "nm1_qc_pass": qc[qc],
    "tx_hq": pn["tx_hq"].ravel()[qc],
    "verified_isbc": pn["verified_isbc"].ravel()[qc],
    "cell_row": pn["cell_row"].ravel()[qc],
    "full_row": np.flatnonzero(qc) + 1,
})
assert np.equal(neurons["cell_row"], np.floor(neurons["cell_row"])).all()
neurons["cell_row"] = neurons["cell_row"].astype("int64")
assert neurons["cell_row"].gt(0).all()
assert neurons["uid"].is_unique and neurons["nm1_qc_pass"].all()
write_table(neurons, "neurons.csv")
uid = neurons["uid"].to_numpy()
display(neurons.groupby("brain", sort=True).agg(
    neurons=("uid", "size"), tx_hq=("tx_hq", "sum")
))

,neurons,tx_hq
brain,,
777552,1035,603
824690,413,233


## 2. `soma_ccf.csv`

Source: `pn.CCF`. Columns are AP, DV, and ML in CCFv3 25 um voxel coordinates.

Coordinates are preserved without rounding, rebasing, or conversion to micrometres.

In [3]:
ccf = np.asarray(pn["CCF"])[qc]
assert np.isfinite(ccf).all()
soma_ccf = pd.DataFrame(ccf, columns=["ap_vox", "dv_vox", "ml_vox"])
soma_ccf.insert(0, "uid", uid)
write_table(soma_ccf, "soma_ccf.csv")

## 3. `projections_ccf_429.csv`

Source: `pn.proj_mat` and `pn.proj_area_names`.

One row per QC neuron; 429 region columns in source order. Values are raw rolony counts. Keep `NA` and VISp columns.

In [4]:
leaf_names = pn["proj_area_names"].ravel().tolist()
leaf_counts = pn["proj_mat"].tocsr()[qc].toarray()
assert len(leaf_names) == len(set(leaf_names)) == 429
assert np.isfinite(leaf_counts).all() and (leaf_counts >= 0).all()
assert np.equal(leaf_counts, np.floor(leaf_counts)).all()
leaf_counts = leaf_counts.astype("int64")
leaf_matrix = pd.DataFrame(leaf_counts, columns=leaf_names)
leaf_matrix.insert(0, "uid", uid)
write_table(leaf_matrix, "projections_ccf_429.csv")

## 4. `projection_targets.csv`

Reference: [xce01_target_projection_matrix.m](../../data/BARseq/scripts/lib/xce01_target_projection_matrix.m).

Defines the fixed 18-target order and matching rules. `match_keys` uses `|` to separate alternatives. CP is excluded.

In [5]:
cortical_targets = [
    "VISpor", "VISpl", "VISli", "VISl", "VISal", "VISrl", "VISa",
    "VISam", "VISpm", "RSPagl", "RSPd", "RSPv", "SSp-bfd",
]
target_spec = pd.DataFrame(
    [(name, "parent", name) for name in cortical_targets] + [
        ("LGd", "parent", "LGd"),
        ("LP", "name", "LP"),
        ("LD", "name", "LD"),
        ("RT", "name", "RT"),
        ("SC", "parent", "SCs|SCm"),
    ],
    columns=["target", "source_field", "match_keys"],
)
target_spec.insert(0, "target_order", np.arange(1, len(target_spec) + 1))
target_names = target_spec["target"].tolist()
assert len(target_names) == len(set(target_names)) == 18
write_table(target_spec, "projection_targets.csv")

### Count target rolonies and split laterality

All cohort somata must lie left of the midline. Ipsi: ML < 223; contra: ML > 233.

The inclusive 223-233 voxel band contributes to total counts only. Counts are not normalised or coverage-masked.

In [6]:
MIDLINE_VOX = 228
MARGIN_VOX = 5
assert (ccf[:, 2] < MIDLINE_VOX - MARGIN_VOX).all()
rolony_tables = pn["rolonies"].ravel()[qc]
source_n_rolonies = pn["n_rolonies"].ravel()[qc]
assert np.array_equal([len(table) for table in rolony_tables], source_n_rolonies)
rolonies = pd.concat([
    table[["parent", "name", "ccf_lr"]].assign(neuron_row=row)
    for row, table in enumerate(rolony_tables)
], ignore_index=True)
assert np.isfinite(rolonies["ccf_lr"]).all()
assert rolonies[["parent", "name"]].notna().all().all()
target_col = np.full(len(rolonies), -1, dtype="int64")
for col, rule in enumerate(target_spec.itertuples(index=False)):
    keys = rule.match_keys.split("|")
    matches = rolonies[rule.source_field].isin(keys).to_numpy()
    assert (target_col[matches] == -1).all(), f"Overlapping rule: {rule.target}"
    if rule.source_field == "name":
        assert not rolonies["parent"].isin(keys).any(), (
            f"Incomplete leaf rule for {rule.target}: target also occurs as a parent"
        )
    target_col[matches] = col

neuron_row = rolonies["neuron_row"].to_numpy()
ml = rolonies["ccf_lr"].to_numpy()
selected = target_col >= 0
total = np.zeros((len(uid), len(target_names)), dtype="int64")
ipsi = np.zeros_like(total)
contra = np.zeros_like(total)
for matrix, mask in (
    (total, selected),
    (ipsi, selected & (ml < MIDLINE_VOX - MARGIN_VOX)),
    (contra, selected & (ml > MIDLINE_VOX + MARGIN_VOX)),
):
    np.add.at(matrix, (neuron_row[mask], target_col[mask]), 1)
assert total.sum() == selected.sum()
assert (total.sum(axis=1) <= source_n_rolonies).all()
assert (ipsi + contra <= total).all()
assert total.sum() - ipsi.sum() - contra.sum() == (
    selected & (ml >= 223) & (ml <= 233)
).sum()
print(f"Neurons with target rolonies: {(total.sum(axis=1) > 0).sum():,}/{len(uid):,}")

Neurons with target rolonies: 1,435/1,448


## 5. `projections_targets_18_total.csv`

All matched target rolonies, including the midline band. Rows follow `neurons.csv`; columns follow `projection_targets.csv`.

In [7]:
total_frame = pd.DataFrame(total, columns=target_names)
total_frame.insert(0, "uid", uid)
write_table(total_frame, "projections_targets_18_total.csv")

## 6. `projections_targets_18_ipsi.csv`

Ipsilateral target rolonies: ML < 223 voxels. Same row and column order as the total matrix.

In [8]:
ipsi_frame = pd.DataFrame(ipsi, columns=target_names)
ipsi_frame.insert(0, "uid", uid)
write_table(ipsi_frame, "projections_targets_18_ipsi.csv")

## 7. `projections_targets_18_contra.csv`

Contralateral target rolonies: ML > 233 voxels. Same row and column order as the total matrix.

Contralateral imaging is incomplete. Zero counts do not establish absence of a contralateral projection; see source metadata in `provenance.json`.

In [9]:
contra_frame = pd.DataFrame(contra, columns=target_names)
contra_frame.insert(0, "uid", uid)
write_table(contra_frame, "projections_targets_18_contra.csv")

## 8. `brain_target_coverage.csv`

One row per brain and target: cohort size, rolony count, projecting-neuron count, and rolonies per neuron.

`usable_both_brains` follows xce01: every brain has >0 rolonies and the minimum/maximum per-neuron rate is >=0.05.

The source identifies unsampled targets in 824690: LGd, LD, SSp-bfd, VISa; LP has limited sampling. `not_flagged` does not certify complete coverage. Matrix zeros remain unchanged.

In [10]:
coverage_rows = []
for brain in sorted(neurons["brain"].unique()):
    brain_mask = neurons["brain"].eq(brain).to_numpy()
    for col, target in enumerate(target_names):
        counts = total[brain_mask, col]
        source_flag = "not_flagged"
        if brain == "824690":
            if target in {"LGd", "LD", "SSp-bfd", "VISa"}:
                source_flag = "unsampled"
            elif target == "LP":
                source_flag = "limited_sampling"
        coverage_rows.append({
            "brain": brain, "target": target, "n_cohort": int(brain_mask.sum()),
            "n_rolonies": int(counts.sum()), "n_neurons": int((counts > 0).sum()),
            "rolonies_per_neuron": float(counts.mean()), "source_flag": source_flag,
        })
coverage = pd.DataFrame(coverage_rows)
rates = coverage.groupby("target")["rolonies_per_neuron"]
coverage["min_max_rate_ratio"] = (
    rates.transform("min") / rates.transform("max").clip(lower=np.finfo(float).eps)
)
coverage["usable_both_brains"] = (
    coverage.groupby("target")["n_rolonies"].transform("min").gt(0)
    & coverage["min_max_rate_ratio"].ge(0.05)
)
write_table(coverage, "brain_target_coverage.csv")

## 9. `taxonomy_classes.csv`

BARseq inferred taxonomy, not the joint taxonomy or a Tasic/ABC reference hierarchy.

Class membership follows [xcs00_taxonomy_map.m](../../data/BARseq/scripts/lib/xcs00_taxonomy_map.m). `NA` is unassigned, not a taxonomy node.

In [11]:
subclass_names = pn["subclass_names"].ravel().tolist()
class_members = {
    "Cortex": subclass_names[:18],
    "Hippocampus": ["CA1", "CA3", "DG", "DG_2", "SUB", "SUBd", "PPP", "gaba_HIP"],
    "Thalamus": ["TH", "PF", "MH", "TRN"],
    "Midbrain": ["MB_1", "MB_2", "MB_Ctx", "MB_RT", "SC", "VTA_SNc", "DR"],
    "Hindbrain": ["PG"],
    "Hypothalamus": ["HY"],
    "Striatum": ["Str"],
    "Amygdala/Olfactory": ["Amyg", "LA", "EPd", "PIR_low_qual"],
    "Nonneuronal": ["Nonneuronal"],
    "Unassigned": ["Low_qual"],
}
class_of_subclass = {
    subclass: parent for parent, members in class_members.items() for subclass in members
}
assert len(subclass_names) == len(set(subclass_names)) == 46
assert sum(map(len, class_members.values())) == len(class_of_subclass) == 46
assert set(class_of_subclass) == set(subclass_names)
classes = pd.DataFrame({
    "class_order": np.arange(1, len(class_members) + 1),
    "class": list(class_members),
})
write_table(classes, "taxonomy_classes.csv")

## 10. `taxonomy_subclasses.csv`

All 46 named subclasses, with class parents. Preserve source indices and curated order from [xcd02_build_taxonomy_index.m](../../data/BARseq/scripts/lib/xcd02_build_taxonomy_index.m).

In [12]:
subclasses = pd.DataFrame({
    "subclass_idx": np.arange(1, len(subclass_names) + 1),
    "subclass": subclass_names,
    "class": [class_of_subclass[name] for name in subclass_names],
})
write_table(subclasses, "taxonomy_subclasses.csv")

## 11. `taxonomy_clusters.csv`

Export the 83 observed clusters with verified subclass parents. Preserve their original `clust_idx` values and source order.

The source lookup has 185 names. The other 102 have no cohort assignments and are not exported. Parent relationships are taken from paired labels, never inferred from cluster names.

In [13]:
all_labels = pd.DataFrame({
    "subclass": pn["subclass"].ravel(),
    "cluster": pn["clust"].ravel(),
})
parent_pairs = all_labels.loc[all_labels["cluster"].ne("NA")].drop_duplicates()
assert parent_pairs["cluster"].is_unique, "A cluster has multiple subclass parents"
assert parent_pairs["subclass"].isin(subclass_names).all()
cluster_names = pn["clust_names"].ravel().tolist()
assert len(cluster_names) == len(set(cluster_names)) == 185
assert parent_pairs["cluster"].isin(cluster_names).all()
cluster_lookup = pd.DataFrame({
    "clust_idx": np.arange(1, len(cluster_names) + 1), "cluster": cluster_names,
})
clusters = cluster_lookup.merge(parent_pairs, on="cluster", how="inner", validate="one_to_one")
clusters = clusters.merge(subclasses, on="subclass", how="left", validate="many_to_one")
clusters = clusters.sort_values("clust_idx").reset_index(drop=True)
assert not clusters.isna().any().any()
write_table(clusters, "taxonomy_clusters.csv")

## 12. `neuron_taxonomy.csv`

Source: plain `pn.subclass`, `pn.clust`, and their indices. One row per QC neuron; no `tx_hq` filtering.

Preserve `NA` labels and index 0 for unassigned cells. These are not taxonomy nodes.

In [14]:
memberships = all_labels.loc[qc].reset_index(drop=True).copy()
memberships.insert(0, "uid", uid)
for field, names in (("subclass_idx", subclass_names), ("clust_idx", cluster_names)):
    values = pn[field].ravel()[qc]
    assert np.equal(values, np.floor(values)).all()
    indices = values.astype("int64")
    assert ((indices >= 0) & (indices <= len(names))).all()
    label_column = "subclass" if field == "subclass_idx" else "cluster"
    assert np.array_equal(np.array(["NA", *names])[indices], memberships[label_column])
    memberships[field] = indices
assigned = memberships["cluster"].ne("NA")
assert memberships.loc[assigned, "cluster"].isin(clusters["cluster"]).all()
assert memberships.loc[assigned, "subclass"].eq(
    memberships.loc[assigned, "cluster"].map(clusters.set_index("cluster")["subclass"])
).all()
assert np.array_equal(
    memberships["subclass_idx"].between(1, 18), neurons["tx_hq"]
)
write_table(memberships, "neuron_taxonomy.csv")

## 13. `harmony_xck12.csv`

Source: `xck12_harmony_predictions.csv.gz`. Tasic cluster and ABC supertype/cluster predictions, including confidence and source annotations.

Match `sample` to QC neuron UIDs and rename it `uid`. Keep only available prediction rows, in cohort order. No confidence or `tx_hq` filter; no predictions are filled in.

In [15]:
def read_cohort_mapping(path: Path) -> pd.DataFrame:
    parts = []
    for chunk in pd.read_csv(
        path, chunksize=250_000, dtype={"sample": "string", "brain": "string"},
        keep_default_na=False,
    ):
        selected_rows = chunk.loc[chunk["sample"].isin(uid)].copy()
        if not selected_rows.empty:
            parts.append(selected_rows)
    assert parts, f"No QC cohort predictions found in {path}"
    frame = pd.concat(parts, ignore_index=True).rename(columns={"sample": "uid"})
    assert frame["uid"].is_unique, f"Duplicate predictions in {path}"
    assert frame["brain"].eq(frame["uid"].map(neurons.set_index("uid")["brain"])).all()
    for column in frame.columns:
        if column.endswith("_conf"):
            confidence = pd.to_numeric(frame[column], errors="raise")
            assert confidence.between(0, 1).all(), f"Invalid confidence: {path}, {column}"
    return neurons[["uid"]].merge(frame, on="uid", how="inner", validate="one_to_one")


mapping12 = read_cohort_mapping(XCK12_PATH)
write_table(mapping12, "harmony_xck12.csv")
print(f"xck12 predictions: {len(mapping12):,}/{len(uid):,} QC neurons")

/tmp/ipykernel_21242/610630524.py:3: DtypeWarning: Columns (8,10,12) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


/tmp/ipykernel_21242/610630524.py:3: DtypeWarning: Columns (8,10,12) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


/tmp/ipykernel_21242/610630524.py:3: DtypeWarning: Columns (8,10,12) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


xck12 predictions: 750/1,448 QC neurons


## 14. `harmony_xck17.csv`

Source: `xck17_harmony_predictions.csv.gz`. ABC subclass/cluster predictions and confidence.

Use the same UID selection as xck12. Keep the two exports separate: their reference labels and confidence columns differ.

In [16]:
mapping17 = read_cohort_mapping(XCK17_PATH)
write_table(mapping17, "harmony_xck17.csv")
print(f"xck17 predictions: {len(mapping17):,}/{len(uid):,} QC neurons")

/tmp/ipykernel_21242/610630524.py:3: DtypeWarning: Columns (8,10) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


/tmp/ipykernel_21242/610630524.py:3: DtypeWarning: Columns (8,10) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/tmp/ipykernel_21242/610630524.py:3: DtypeWarning: Columns (8,10) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


/tmp/ipykernel_21242/610630524.py:3: DtypeWarning: Columns (8,10) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


/tmp/ipykernel_21242/610630524.py:3: DtypeWarning: Columns (8,10) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


xck17 predictions: 834/1,448 QC neurons


/tmp/ipykernel_21242/610630524.py:3: DtypeWarning: Columns (8,10) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


## 15. `subclass_colours.csv`

Hardcoded name-to-colour mapping for the 18 cortical subclasses. The other 28 subclasses have blank colour fields.

Values reproduce [s53_xcac00_dump_xcs03_palette.m](../../data/BARseq/scripts/scripts/s53_xcac00_dump_xcs03_palette.m): `hclrainbow(18, [0 330])`, chroma 50, luminance 70, stride 7, rounded to 8-bit RGB.

Match by subclass name, not position: the palette order differs from the curated taxonomy order.

In [17]:
SUBCLASS_COLOURS = {
    "L2_3_IT": "#FE85AD",
    "L4_5_IT": "#7BBB6A",
    "L5_IT": "#5FAFFF",
    "L5_ET": "#EC975F",
    "L6_IT": "#00CFCD",
    "L6_CT": "#E38ED9",
    "L6b": "#9EB457",
    "NP": "#00BCFF",
    "Car3": "#FA8D74",
    "RSP_ML": "#00C4A4",
    "RSP_UL": "#C499F0",
    "ENT_ML": "#BCAC4E",
    "ENT_UL1": "#00CEFF",
    "ENT_UL2": "#FF868F",
    "ENT_UL3": "#51BF83",
    "ENT_UL4": "#99A4FF",
    "gaba_Ctx_CGE": "#D6A251",
    "gaba_Ctx_MGE": "#00D3EB",
}
assert set(SUBCLASS_COLOURS) == set(class_members["Cortex"])
assert len(set(SUBCLASS_COLOURS.values())) == 18
colours = subclasses[["subclass_idx", "subclass"]].copy()
colours["hex_colour"] = colours["subclass"].map(SUBCLASS_COLOURS)
assert colours["hex_colour"].notna().sum() == 18
assert colours["hex_colour"].dropna().str.fullmatch(r"#[0-9A-F]{6}").all()
write_table(colours, "subclass_colours.csv")

## 16. `provenance.json`

Source metadata, QC thresholds, extraction rules, software versions, file hashes, and output schemas.

This package stores `meta` at the MAT file's top level, not inside `pn`. Preserve it here, including contralateral coverage limitations.

In [18]:
def numpy_json(value: object) -> object:
    if isinstance(value, np.void) and value.dtype.names:
        return {name: value[name] for name in value.dtype.names}
    if isinstance(value, np.ndarray):
        return value.reshape(-1)[0] if value.size == 1 else value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Unsupported metadata type: {type(value).__name__}")


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


reference_paths = [SOURCE_ROOT / relative for relative in (
    "lib/xca02_load_proj_neurons.m",
    "lib/xce01_target_projection_matrix.m",
    "lib/xcd02_build_taxonomy_index.m",
    "lib/xcs00_taxonomy_map.m",
    "scripts/s53_xcac00_dump_xcs03_palette.m",
    "lib/external/hclrainbow.m",
    "lib/external/colorspace.m",
)]
provenance = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "code/BARseq/BARseq_data_extraction_01.ipynb",
    "sources": [
        {"path": path.relative_to(REPO_ROOT).as_posix(), "sha256": file_sha256(path)}
        for path in (MAT_PATH, XCK12_PATH, XCK17_PATH, *reference_paths)
    ],
    "software": {name: version(name) for name in ("mat-io", "numpy", "pandas", "scipy")},
    "source_meta": source_meta,
    "cohort": {
        "selection": "pn.nm1_qc_pass; no target-count or tx_hq filter",
        "n_source": len(uid_all), "n_qc": len(uid),
        "n_tx_hq": int(neurons["tx_hq"].sum()),
        "n_with_target_rolonies": int((total.sum(axis=1) > 0).sum()),
        "n_matched_rolonies": len(rolonies),
        "n_target_rolonies": int(total.sum()),
        "counts_by_brain": neurons.groupby("brain").size().to_dict(),
    },
    "coordinates": {
        "reference": "CCFv3", "voxel_size_um": 25,
        "axis_order": ["AP", "DV", "ML"], "values": "source voxel coordinates, unchanged",
        "full_row_and_cell_row_index_base": 1,
    },
    "projections": {
        "unit": "matched rolony count", "coverage_mask_applied": False,
        "target_rules": target_spec.to_dict(orient="records"),
        "midline_vox": MIDLINE_VOX, "margin_vox": MARGIN_VOX,
        "ipsi_rule": "ccf_lr < 223", "contra_rule": "ccf_lr > 233",
        "midline_band": "223 <= ccf_lr <= 233; total only",
        "usable_both_brains_rule": "all counts > 0 and min/max per-neuron rate >= 0.05",
        "coverage_warning": "Zero counts do not distinguish no projection from no sampling",
    },
    "taxonomy": {
        "source": "inferred, plain pn fields; not *_joint",
        "n_subclasses": len(subclasses), "n_source_cluster_names": len(cluster_names),
        "n_exported_clusters": len(clusters),
        "cluster_selection": "observed clusters with verified subclass parents",
        "unassigned_label": "NA", "unassigned_index": 0,
        "colours": "18 cortical reference colours; remaining subclasses uncoloured",
    },
    "harmony": {
        "selection": "sample in QC UIDs; no confidence or tx_hq filter",
        "n_xck12": len(mapping12), "n_xck17": len(mapping17),
        "missing_predictions": "omitted rows; no imputation",
    },
    "outputs": {
        name: {**schema, "sha256": file_sha256(OUTPUT_ROOT / name)}
        for name, schema in output_tables.items()
    },
}
provenance_text = json.dumps(provenance, default=numpy_json, indent=2, allow_nan=False)
(OUTPUT_ROOT / "provenance.json").write_text(provenance_text + "\n", encoding="utf-8")

20937

## Validate written files

Check row alignment, count round-trips, taxonomy links, and source-package cohort totals. These checks target the supplied package.

In [19]:
assert len(uid_all) == 1512 and len(neurons) == 1448
assert neurons.groupby("brain").size().to_dict() == {"777552": 1035, "824690": 413}
assert (total.sum(axis=1) > 0).sum() == 1435
assert len(rolonies) == 47157
assert len(subclasses) == 46 and len(clusters) == 83
assert memberships.loc[assigned, "clust_idx"].isin(clusters["clust_idx"]).all()
assert len(mapping12) == 750 and len(mapping17) == 834
gap_rows = coverage[coverage["source_flag"].eq("unsampled")]
assert len(gap_rows) == 4 and gap_rows["n_rolonies"].eq(0).all()
assert coverage.loc[coverage["source_flag"].eq("limited_sampling"), "n_rolonies"].eq(4).all()

for name, schema in output_tables.items():
    saved = pd.read_csv(OUTPUT_ROOT / name, keep_default_na=False, dtype={"uid": "string"})
    assert len(saved) == schema["rows"] and saved.columns.tolist() == schema["columns"], name
    if "uid" in saved:
        assert saved["uid"].is_unique and saved["uid"].isin(uid).all(), name
        expected_uid = uid if len(saved) == len(uid) else uid[np.isin(uid, saved["uid"])]
        assert np.array_equal(saved["uid"], expected_uid), name

for filename, expected in (
    ("projections_ccf_429.csv", leaf_counts),
    ("projections_targets_18_total.csv", total),
    ("projections_targets_18_ipsi.csv", ipsi),
    ("projections_targets_18_contra.csv", contra),
):
    saved = pd.read_csv(OUTPUT_ROOT / filename, keep_default_na=False)
    assert np.array_equal(saved.iloc[:, 1:].to_numpy(), expected), filename

saved_provenance = json.loads((OUTPUT_ROOT / "provenance.json").read_text(encoding="utf-8"))
assert saved_provenance["source_meta"]["nm1_qc"]["n_pass"] == len(uid)
for name, schema in saved_provenance["outputs"].items():
    assert file_sha256(OUTPUT_ROOT / name) == schema["sha256"], name
summary = pd.DataFrame([
    {"file": name, "rows": schema["rows"], "columns": len(schema["columns"])}
    for name, schema in output_tables.items()
])
display(summary)
print(f"Validated {len(output_tables)} CSV files and provenance.json in {OUTPUT_ROOT}")

,file,rows,columns
0,neurons.csv,1448,7
1,soma_ccf.csv,1448,4
2,projections_ccf_429.csv,1448,430
3,projection_targets.csv,18,4
4,projections_targets_18_total.csv,1448,19
5,projections_targets_18_ipsi.csv,1448,19
6,projections_targets_18_contra.csv,1448,19
7,brain_target_coverage.csv,36,9
8,taxonomy_classes.csv,10,2
9,taxonomy_subclasses.csv,46,3


Validated 15 CSV files and provenance.json in /workspaces/ConnectsCommonConnectivity/data/BARseq/extracted
